## Milestone 1 Exploration
### Jennifer Tsang, Nicole Link

In [1]:
#os.getcwd()
#%cd "/Users/Nicole/MDS/575/DSCI_575_project_jt8919_nicolelink33"
#%pwd

In [ ]:
import sys
import os
import json
from pprint import pprint
from pathlib import Path
import re
import pandas as pd
import pyarrow as pa
import duckdb
import numpy as np
import pyarrow.parquet as pq
import requests
from tqdm import tqdm
from langchain_community.document_loaders import DataFrameLoader
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
import pickle

project_root = Path.cwd().parent 
os.chdir(project_root)

from src.utils import simple_tokenize
from src.bm25 import search_with_scores, display_top_results

In [3]:
DATA_DIR = Path("data")
CATEGORY = "Arts_Crafts_and_Sewing"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"
OUTPUT_FILE  = DATA_DIR / f"{CATEGORY}_merged.parquet"
bm25_path = 'models/bm25_metadata_index_2.pkl'

#reviews_path = '../data/raw/Arts_Crafts_and_Sewing.jsonl'
#meta_path = '../data/raw/meta_Arts_Crafts_and_Sewing.jsonl'
#cleaned_reviews_path = '../data/processed/cleaned_reviews.parquet'
#cleaned_meta_path = '../data/processed/cleaned_meta.parquet'
#hybrid_parquet_path = '../data/processed/hybrid_documents.parquet'
#bm25_path = '../models/bm25_metadata_index.pkl'

In [4]:
print(REVIEWS_URL)

https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Arts_Crafts_and_Sewing.jsonl.gz


## Loading and Merging the Datasets

### Data Loading

In [5]:
c2 = duckdb.connect()

In [6]:
head_reviews = c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
head_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1.0,Received damaged. Has hole in mold!,[[VIDEOID:2a7ad2a91afb1e17ff4a1c143f7e10a2]] R...,[{'small_image_url': 'https://m.media-amazon.c...,B095RKB9N3,B095RXT585,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1661111719157,0,True
1,3.0,3rd one arrived scratched/dented.,3rd one that arrived damaged! I give up!,[{'small_image_url': 'https://m.media-amazon.c...,B08PNNKNSQ,B07QXN7TMP,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1655614875170,0,True
2,1.0,Abominable. NOT COLOR SHIFT! False Advertising!,These are regular mica powders! One appears t...,[{'small_image_url': 'https://m.media-amazon.c...,B094DH11SH,B094DH11SH,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1654510930828,0,True
3,2.0,Used 2x in 6 weeks dry as a bone.,[[VIDEOID:8644a01103c90ab8f83af816b83881be]] L...,[{'small_image_url': 'https://m.media-amazon.c...,B089YSSFD6,B087D548ZY,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1648170476726,0,True
4,1.0,Replaced & still Defective stamps!,Lowered from a 3 star review to a 1 star bc th...,[{'small_image_url': 'https://m.media-amazon.c...,B08Z47RG56,B08Z47RG56,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1644619507972,1,True


In [7]:
# executes right over the internet -- i just want five rows to preview so doesn't take long
head_meta = c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Amazon Home,Trapp Home Fragrance Wax Melts - No. 13 Bob's ...,4.4,108,[Includes (2) No. 13 Bob's Flower Shoppe Trapp...,[The No. 13 Bob's Flower Shoppe Home Fragrance...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Trapp,"[Arts, Crafts & Sewing, Crafting, Candle Makin...","{'Product Dimensions': '""6.75 x 3.4 x 1 inches...",B013W3MPCW,None,None,<NA>
1,"Arts, Crafts & Sewing",JESEP YONG 4packs Plastic Organizer Box 24 Gri...,4.3,103,[MATERIAL & COLOR: These storage boxes are mad...,[SIZE & PACKING: the outer box size is 7.5'' L...,12.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'craft organizers and storage', 'ur...",JESEP YONG,"[Arts, Crafts & Sewing, Organization, Storage ...","{'Brand': '""JESEP YONG""', 'Color': '""Clear""', ...",B09B59XWTG,None,None,<NA>
2,Pet Supplies,"JIESHU 2-Pack Foam Holders for centerpieces, B...",3.6,15,[The 6 suction cups at the bottom of the foam ...,[Size: 9.5 * 4.5 * 2.7 inches Ideal for manual...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],JIESHU,"[Arts, Crafts & Sewing, Crafting, Floral Arran...","{'Product Dimensions': '""9.5 x 4.5 x 2.7 inche...",B0915N2NX1,None,None,<NA>
3,"Arts, Crafts & Sewing",Metal Swivel Lobster Claw Clasp Split Ring Key...,3.2,35,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],NaN,"[Arts, Crafts & Sewing, Beading & Jewelry Maki...","{'Product Dimensions': '""4.1 x 3.7 x 0.6 inche...",B008SNYJSU,None,None,<NA>
4,"Arts, Crafts & Sewing",Sizzix Birdcage Favor Thinlits Gift Box by Dav...,4.6,464,[Thinlits dies allow you to cut intricate desi...,[Thinlits create dazzling detailed shapes for ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Sizzix,"[Arts, Crafts & Sewing, Scrapbooking & Stampin...","{'Product Dimensions': '""5.38 x 4.38 x 0.13 in...",B07Z9FH9PR,None,None,<NA>


#### Download and convert to parquet

In [8]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{REVIEWS_URL}')  LIMIT 20000)
      TO 'data/raw/reviews_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

In [9]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{META_URL}') LIMIT 20000)
      TO 'data/raw/meta_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

#### Merge the two files

In [10]:
c2.execute("""
    COPY (
        SELECT r.rating, r.title, r.text, r.parent_asin, r.verified_purchase,
           r.helpful_vote,
           m.title AS product_title, m.price,
           m.average_rating, m.main_category, m.store
        FROM read_parquet('data/raw/reviews_raw.parquet') r
        LEFT JOIN read_parquet('data/raw/meta_raw.parquet') m USING (parent_asin)
    )
    TO 'data/merged.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [11]:
c2.execute(f"SELECT * FROM read_parquet('data/merged.parquet')").df()


,rating,title,text,parent_asin,verified_purchase,helpful_vote,product_title,price,average_rating,main_category,store
0,5.0,"Beautiful, heavily textured vinyl",I really like this assortment of metallic viny...,B0BW3X9XT4,False,0,Holographic Vinyl Black Rainbow Permanent Viny...,13.90,4.6,Amazon Home,GIRAFVINYL
1,2.0,Too small and seem to be better for rings,I've had a hard time using these for bracelets...,B0058EDF8C,True,0,"Eurotool Pliers, Silver",21.99,4.1,"Arts, Crafts & Sewing",EURO TOOL
2,5.0,Awesome product.,I absolutely loved being able to buy feathers ...,B07NP9VFZ3,True,1,wanjin Duck Goose Feathers Trim Fringe Craft F...,14.99,4.6,"Arts, Crafts & Sewing",Wanjin
3,5.0,Very true to color in picture. Fast shipping. ...,Beautiful beads very true to picture. Wish I h...,B00J9F6DEO,True,5,BEADNOVA 8mm White Clear Crystal Quartz Gemsto...,11.39,4.7,"Arts, Crafts & Sewing",BEADNOVA
4,5.0,Five Stars,I love this small size sketch pad. Perfect si...,B088VVG9HN,True,0,"Strathmore 300 Series Sketch Paper Pad, Glue B...",24.61,4.7,"Arts, Crafts & Sewing",Strathmore
...,...,...,...,...,...,...,...,...,...,...,...
19995,5.0,Five Stars,Great pen set!! I would buy this over and over.,B01GLS0C2K,True,1,NaN,NaN,NaN,NaN,NaN
19996,5.0,Wow,Beautiful product. Very secure. Gave a few awa...,B017K4AFF8,True,0,NaN,NaN,NaN,NaN,NaN
19997,5.0,Perfect for my needs.,Melts very easily. Great value at this size,B001QMY0RU,True,0,NaN,NaN,NaN,NaN,NaN
19998,5.0,Great selection of zippers for the price!,so far I’m very well pleased with the value I ...,B0756TPB4S,True,2,NaN,NaN,NaN,NaN,NaN


### Create Stratified Sample

In [12]:
SAMPLE_PER_STRATUM = 50    # reviews to keep per stratum cell (floor guarantee)
MIN_TEXT_LEN       = 20    # drop near-empty reviews (chars)
SHORT_MAX          = 100   # short: text < SHORT_MAX chars
MEDIUM_MAX         = 500   # medium: SHORT_MAX ≤ text < MEDIUM_MAX chars
                        # long: text ≥ MEDIUM_MAX chars

OUTPUT = 'data/processed/stratified_sample.parquet'

c2.execute(f"""
COPY (
WITH labelled AS (
    SELECT
        text, rating, verified_purchase, helpful_vote,
        parent_asin,
        product_title, price,
        average_rating, main_category,

        CASE
            WHEN average_rating >= 4.6 THEN '4.6-5.0'
            WHEN average_rating >= 4.4 THEN '4.4-4.5'
            WHEN average_rating >= 4.1 THEN '4.1-4.3'
            WHEN average_rating >= 3.7 THEN '3.7-4.0'
            WHEN average_rating >= 3.1 THEN '3.1-3.6'
            ELSE                              '<=3.0'
        END AS rating_bucket,

        CASE
            WHEN LENGTH(text) < {SHORT_MAX}  THEN 'short'
            WHEN LENGTH(text) < {MEDIUM_MAX} THEN 'medium'
            ELSE                                    'long'
        END AS len_tier

    FROM read_parquet('data/merged.parquet')
    WHERE text IS NOT NULL
    AND LENGTH(text) >= {MIN_TEXT_LEN}
    AND average_rating IS NOT NULL
),

-- Step 1: one review per product per cell: prevents a single popular product flooding a stratum, you can redefine
one_per_product AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY rating_bucket, len_tier, verified_purchase, parent_asin
            ORDER BY helpful_vote DESC, random()
        ) AS product_rank
    FROM labelled
),

-- Step 2: rank within each stratum cell, helpful reviews first
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY rating_bucket, len_tier, verified_purchase
            ORDER BY helpful_vote DESC, random()
        ) AS stratum_rank
    FROM one_per_product
    WHERE product_rank = 1
)

SELECT * EXCLUDE (product_rank, stratum_rank)
FROM ranked
WHERE stratum_rank <= {SAMPLE_PER_STRATUM}
)
TO '{OUTPUT}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

### Convert Parquet to LangChain Documents

In [13]:
# 1. Load your newly minted stratified sample
df_sample = pd.read_parquet(OUTPUT)

# 2. Fill any random missing text values to prevent concatenation crashes
text_columns = ['product_title', 'main_category', 'text']
for col in text_columns:
    if col in df_sample.columns:
        df_sample[col] = df_sample[col].fillna("")

# 3. Engineer the hybrid page_content string
df_sample['page_content'] = (
    "Product Title: " + df_sample['product_title'] + "\n" +
    "Category: " + df_sample['main_category'] + "\n" +
    "Review Text: " + df_sample['text']
)

# 4. Drop the redundant text columns so they don't clutter the metadata
df_clean = df_sample.drop(columns=['product_title', 'main_category', 'text'])

# 5. Convert the DataFrame into LangChain Documents
documents = []
for _, row in df_clean.iterrows():
    # The engineered string becomes the searchable content
    content = row['page_content']
    
    # Every other column (rating, price, helpful_vote, rating_bucket, len_tier) 
    # gets scooped up into the metadata dictionary!
    metadata = row.drop('page_content').to_dict()
    
    # Create and append the Document
    doc = Document(page_content=content, metadata=metadata)
    documents.append(doc)

print(f"Successfully created {len(documents)} LangChain Documents!")
print("\n--- Let's peek at the first one ---")
print(f"CONTENT:\n{documents[0].page_content}")
print(f"METADATA:\n{documents[0].metadata}")

Successfully created 641 LangChain Documents!

--- Let's peek at the first one ---
CONTENT:
Product Title: Blulu Heat Erasable Fabric Marking Pens with 8 Refills for Tailors Sewing, and Quilting Dressmaking, 4 Colors Heat Erasable Pens for Various Colors of Fabrics (12)
Category: Office Products
Review Text: Work really well, disappear completely with heat
METADATA:
{'rating': 3.0, 'verified_purchase': True, 'helpful_vote': 0, 'parent_asin': 'B07M6JW3PY', 'price': nan, 'average_rating': 3.6, 'rating_bucket': '3.1-3.6', 'len_tier': 'short'}


## Data Overview:

In [14]:
print(df_sample.info())
print(df_sample.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 641 entries, 0 to 640
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   text               641 non-null    str    
 1   rating             641 non-null    float64
 2   verified_purchase  641 non-null    bool   
 3   helpful_vote       641 non-null    int64  
 4   parent_asin        641 non-null    str    
 5   product_title      641 non-null    str    
 6   price              525 non-null    float64
 7   average_rating     641 non-null    float64
 8   main_category      641 non-null    str    
 9   rating_bucket      641 non-null    str    
 10  len_tier           641 non-null    str    
 11  page_content       641 non-null    str    
dtypes: bool(1), float64(3), int64(1), str(7)
memory usage: 866.8 KB
None
['text', 'rating', 'verified_purchase', 'helpful_vote', 'parent_asin', 'product_title', 'price', 'average_rating', 'main_category', 'rating_bucket', 'len_tier',

#### Rating Distribution

In [15]:
# EDA example duckdb Rating distribution
c2.execute("""
    SELECT
        rating,
        COUNT(*) AS count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percent
    FROM read_parquet('data/merged.parquet')
    GROUP BY 1
    ORDER BY 1
""").df()

,rating,count,percent
0,1.0,895,4.47
1,2.0,676,3.38
2,3.0,1259,6.30
3,4.0,2503,12.52
4,5.0,14667,73.33


## Tokenize Function: ****** moved to src/utils.py

In [16]:
# # simple tokenization function
# def simple_tokenize(text):
#     # Lowercase everything
#     text = text.lower()
#     # Remove everything except letters, numbers, spaces, and hyphens
#     text = re.sub(r"[^a-z0-9\s-]", "", text)
#     # Split into a list of words
#     return text.split()

## BM25 Search    *** moved to bm25.py

In [ ]:
# 3. The Memory-Safe BM25 Builder

# It will pull documents one by one, tokenize them, and build the math model
retriever = BM25Retriever.from_documents(
    documents,
    preprocess_func=simple_tokenize
)

retriever.k = 3

# Save the massive mathematical index to your hard drive
with open(bm25_path, 'wb') as f:
    pickle.dump(retriever, f)
    
print(f"Index built and safely saved to {bm25_path}!")

Index built and safely saved to models/bm25_metadata_index_2.pkl!


In [18]:
# Test it to make sure it works!
results = retriever.invoke("pink yarn")
print(f"\nTop match: {results[0].page_content}")


Top match: Product Title: Lion Brand Yarn 5800-526 Martha Stewart Glitter Eyelash Yarn, Brownstone
Category: Arts, Crafts & Sewing
Review Text: This pretty, sparkly eyelash yarn  adds a lovely, glamorous  look to to various projects.  The yarn has a touch of delicate pink, and also reflects sparkly rainbow colors.  I am just learning to work with eyelash yarn, but can say it adds a lovely accent to a hat, or scarf.  It can be a bit tricky to work with at first, but you will pick it up as you go along.  For me, it works better in knit projects then as a crochet yarn.  Make sure you note the weight and yardage.  These are tiny skeins, so don't be taken by surprise by that.  It is a bulky yarn, so that is another thing to consider in picking a project.  I like it.


#### BM25 Retrieval Score

In [ ]:
# def search_with_scores(retriever, query, k=3):
#     """
#     Runs a BM25 search and returns the top K documents along with their exact scores.
#     """
#     # 1. Tokenize the query using the exact same function you used to build the index
#     tokenized_query = retriever.preprocess_func(query)
    
#     # 2. Get the raw BM25 scores for EVERY document in your entire dataset
#     all_scores = retriever.vectorizer.get_scores(tokenized_query)
    
#     # 3. Find the index positions of the top 'k' highest scores using NumPy
#     top_k_indices = np.argsort(all_scores)[::-1][:k]
    
#     # 4. Match the highest scores back to their original LangChain Documents
#     results = []
#     for idx in top_k_indices:
#         doc = retriever.docs[idx]
#         score = all_scores[idx]
#         results.append((doc, score))
        
#     return results


In [20]:
# --- Let's test it out! ---
query = "green yarn"
scored_results = search_with_scores(retriever, query, k=3)

print(f"Top {retriever.k} Scored Results for: '{query}'\n" + "-"*50)

for rank, (doc, score) in enumerate(scored_results, 1):
    print(f"Rank {rank} | BM25 Score: {score:.3f}")
    print(f"Rating Bucket: {doc.metadata.get('rating_bucket')} | Length: {doc.metadata.get('len_tier')}")
    print(f"Content snippet: {doc.page_content[:120]}...\n")

Top 3 Scored Results for: 'green yarn'
--------------------------------------------------
Rank 1 | BM25 Score: 9.667
Rating Bucket: 4.1-4.3 | Length: medium
Content snippet: Product Title: T-Shirt Yarn Recycled 130 Yards 1.5 lb Bulky Yarn│Jersey Yarn│Fabric Yarn │T Shirt Yarn for Crochet │ Kni...

Rank 2 | BM25 Score: 5.977
Rating Bucket: 4.6-5.0 | Length: short
Content snippet: Product Title: Lion Brand Yarn Heartland Yarn for Crocheting, Knitting, and Weaving, Multicolor Yarn, 1-Pack, Hot Spring...

Rank 3 | BM25 Score: 5.932
Rating Bucket: 4.6-5.0 | Length: medium
Content snippet: Product Title: (3 Pack) Lion Brand Yarn Vanna's Choice Yarn, Fisherman
Category: Amazon Home
Review Text: it's alright f...



In [ ]:
# def display_top_results(scored_results):
#     print("Top 3 Search Results\n" + "="*50)
    
#     # Slice the list to ensure we only process a maximum of 3 results
#     for rank, (doc, score) in enumerate(scored_results[:3], 1):
        
#         # 1. Parse the title and text out of the page_content string
#         # Our format was: Title \n Category \n Text
#         lines = doc.page_content.split('\n')
#         product_title = lines[0].replace("Product Title: ", "")
#         review_text = lines[2].replace("Review Text: ", "")
        
#         # 2. Truncate the review text to ~200 characters
#         if len(review_text) > 200:
#             truncated_review = review_text[:200].rsplit(' ', 1)[0] + "..."
#         else:
#             truncated_review = review_text
            
#         # 3. Format the rating as visual stars
#         # We grab the rating from the metadata and convert the float to an integer
#         rating = int(doc.metadata.get('rating', 0))
#         stars = "⭐" * rating
        
#         # 4. Display the formatted result
#         print(f"#{rank} | Score: {score:.2f}")
#         print(f"Product: {product_title}")
#         print(f"Rating:  {stars} ({rating}/5)")
#         print(f"Review:  {truncated_review}")
#         print("-" * 50)


In [22]:
# --- How to run it ---
# Assuming you already have 'scored_results' from your search
display_top_results(scored_results)

Top 3 Search Results
#1 | Score: 9.67
Product: T-Shirt Yarn Recycled 130 Yards 1.5 lb Bulky Yarn│Jersey Yarn│Fabric Yarn │T Shirt Yarn for Crochet │ Knitting Tshirt Yarn │ Home Decor DYI Supply │ Recycled Yarn │Trapillo (Apple Green)
Rating:  ⭐⭐⭐⭐ (4/5)
Review:  Using for mask straps. Working great. I just wish that it were a little more soft and that it came in black as well as white!
--------------------------------------------------
#2 | Score: 5.98
Product: Lion Brand Yarn Heartland Yarn for Crocheting, Knitting, and Weaving, Multicolor Yarn, 1-Pack, Hot Springs
Rating:  ⭐⭐⭐⭐⭐ (5/5)
Review:  Great yarn, crochets easily...great color.
--------------------------------------------------
#3 | Score: 5.93
Product: (3 Pack) Lion Brand Yarn Vanna's Choice Yarn, Fisherman
Rating:  ⭐⭐⭐ (3/5)
Review:  it's alright for american yarn. i don't understand why european yarn is so beautiful and all we can get in america is the same type of yarn for many years...there is no progress.
--------------

## Justification of Removed Fields:


### User Reviews:
#### Fields to Keep:
- rating - float - Rating of the product (from 1.0 to 5.0).
- title - str - Title of the user review.
- text - str - Text body of the user review.
- parent_asin - str - Parent ID of the product. Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. Please use parent ID to find product meta.


#### Fields to Remove:
- asin - str - ID of the product.
- images - list - Images that users post after they have received the product. Each image has different sizes (small, medium, large), represented by the small_image_url,medium_image_url, and large_image_url respectively.
- user_id - str - ID of the reviewer
- timestamp - int - Time of the review (unix time)
- verified_purchase - bool- User purchase verification
- helpful_vote - int - Helpful votes of the review



### Metadata
#### Fields to Keep:
- main_category - str - Main category (i.e., domain) of the product.
- title - str - Name of the product.
- average_rating - float - Rating of the product shown on the product page.
- rating_number - int - Number of ratings in the product.
- features - list - Bullet-point format features of the product.
- description - list - Description of the product.
- price - float - Price in US dollars (at time of crawling).
- store - str - Store name of the product.
- categories - list - Hierarchical categories of the product.
- details - dict - Product details, including materials, brand, sizes, etc.
- parent_asin - str - Parent ID of the product.
- bought_together	- list - Recommended bundles from the websites.

#### Fields to Remove:
- images - list - Images of the product. Each image has different sizes (thumb, large, hi_res). The “variant” field shows the position of image.
- videos - list - Videos of the product including title and url.



#### Description/Justification
We chose to use the fields "Product Title", "Category", "Product Features", "Review Title" and "Review Text", in our retrieval process. The first two columns are most useful for the BM25 search method, as we can match exact keywords in the product title and category. Columns "Product Features", "Review Title" and "Review Text" are more useful for the semantic search, as they include more descriptive text. 

We chose to remove the following columns: "asin", "images", "user_id", "timestamp", "verified_purchase", "helpful_vote" from the Reviews dataset. 

## Text Preprocessing Decisions

To clean and preprocess documents (e.g., tokenization) and convert documents into searchable representations, we will perform:
- lowercase
- remove white space
- split words into list

# USELESS

In [23]:
# # 2. The Custom Lazy Parquet Loader
# def lazy_load_parquet(file_path, batch_size=10000):
#     """
#     Streams a Parquet file in chunks and yields LangChain Documents one by one.
#     This prevents the entire dataset from loading into RAM at once.
#     """
#     print(f"Streaming documents from {file_path}...")
#     parquet_file = pq.ParquetFile(file_path)
    
#     for batch in parquet_file.iter_batches(batch_size=batch_size):
#         # Convert just this small batch to pandas
#         df_chunk = batch.to_pandas()
        
#         # Yield one Document at a time
#         for _, row in df_chunk.iterrows():
#             content = row['title']
#             # Everything else in the row becomes the metadata dictionary
#             metadata = row.drop('title').to_dict()
            
#             yield Document(page_content=content, metadata=metadata)



In [24]:
# with open(reviews_path, 'r') as fp:
#     for line in fp:
#         pprint(json.loads(line.strip()))
#         break

In [25]:
# with open(meta_path, 'r') as fp:
#     for line in fp:
#         pprint(json.loads(line.strip()))
#         break

## Convert JSON to Parquet

In [26]:
# def convert_jsonl_to_parquet(jsonl_path, parquet_path, chunk_size=100000):
#     print(f"Starting conversion of {jsonl_path} to {parquet_path}...")
    
#     writer = None 
    
#     # Process the large file in chunks to save RAM
#     for chunk in pd.read_json(jsonl_path, lines=True, chunksize=chunk_size):

#         for col in chunk.columns:
#             # If pandas sees a column as 'object', it might contain lists or dicts
#             if chunk[col].dtype == 'object':
#                 # Convert lists and dicts into JSON strings. Leave everything else (strings, NaNs) as-is.
#                 chunk[col] = chunk[col].apply(
#                     lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x
#                 )
            
#         # Convert to PyArrow Table
#         table = pa.Table.from_pandas(chunk)
        
#         # Write to Parquet
#         if writer is None:
#             writer = pq.ParquetWriter(parquet_path, table.schema)
            
#         writer.write_table(table)
#         print(f"Processed {len(chunk)} rows...")

#     if writer:
#         writer.close()
        
#     print("Conversion complete! Your Parquet file is ready.")

In [27]:
# convert_jsonl_to_parquet(reviews_path, cleaned_reviews_path)


### Convert Metadata to Parquet

In [28]:
# def convert_jsonl_to_parquet(jsonl_path, parquet_path, chunk_size=100000):
#     print(f"Starting conversion of {jsonl_path} to {parquet_path}...")
    
#     writer = None 
#     master_schema = None
    
#     # Process the large file in chunks to save RAM
#     for chunk in pd.read_json(jsonl_path, lines=True, chunksize=chunk_size):

#         # Clean price column
#         if 'price' in chunk.columns:
#             # 1. Convert to string so we can strip out any potential ‘$’ or ‘,’ characters
#             chunk['price'] = chunk['price'].astype(str).str.replace(r'[\$,]', '', regex=True)
#             # 2. Force to numeric. Anything that isn’t a number (like ‘-’) becomes NaN
#             chunk['price'] = pd.to_numeric(chunk['price'], errors='coerce')

#         # Stringify complex nested objects
#         for col in chunk.columns:
#             # If pandas sees a column as 'object', it might contain lists or dicts
#             if chunk[col].dtype == 'object':
#                 # Convert lists and dicts into JSON strings. Leave everything else (strings, NaNs) as-is.
#                 chunk[col] = chunk[col].apply(
#                     lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x
#                 )
            
#         # Convert to PyArrow Table
#         table = pa.Table.from_pandas(chunk)
        
#         # Write to Parquet - with schema casting
#         if writer is None:
#             master_schema = table.schema
#             writer = pq.ParquetWriter(parquet_path, master_schema)
#         else:
#             table = table.cast(master_schema)
            
#         writer.write_table(table)
#         print(f"Processed {len(chunk)} rows...")

#     if writer:
#         writer.close()
        
#     print("Conversion complete! Your Parquet file is ready.")

In [29]:
# convert_jsonl_to_parquet(meta_path, cleaned_meta_path)

### Merge Reviews and Metadata

In [30]:
# def create_hybrid_parquet(reviews_parquet, meta_parquet, output_parquet, chunk_size=100000):
#     print("Starting hybrid merge process...")
    
#     # 1. Load ONLY the required metadata columns into memory
#     # By dropping massive columns like 'description' or 'images', we save huge amounts of RAM
#     meta_cols = ['parent_asin', 'title', 'main_category', 'features']
#     df_meta = pd.read_parquet(meta_parquet, columns=meta_cols)
    
#     # Rename the meta title so it doesn't clash with the review title later
#     df_meta = df_meta.rename(columns={'title': 'title_meta'})
    
#     writer = None
#     master_schema = None
    
#     # 2. Stream the massive reviews Parquet file in chunks
#     parquet_file = pq.ParquetFile(reviews_parquet)
    
#     for batch in parquet_file.iter_batches(batch_size=chunk_size):
#         # Convert the PyArrow batch into a pandas DataFrame for easy merging
#         chunk = batch.to_pandas()
        
#         # 3. Merge this specific chunk with the metadata
#         df_merged = pd.merge(
#             chunk, 
#             df_meta, 
#             on='parent_asin', 
#             how='left'
#         )
        
#         # Rename the review title for clarity
#         df_merged = df_merged.rename(columns={'title': 'title_review'})
        
#         # 4. Fill NaNs to prevent string concatenation errors
#         text_columns = ['title_meta', 'main_category', 'title_review', 'text', 'features']
#         for col in text_columns:
#             if col in df_merged.columns:
#                 df_merged[col] = df_merged[col].fillna("")
                
#         # 5. Construct the LangChain page_content string
#         df_merged['page_content'] = (
#             "Product Title: " + df_merged['title_meta'] + "\n" +
#             "Category: " + df_merged['main_category'] + "\n" +
#             "Product Features: " + df_merged['features'].astype(str) + "\n" +
#             "Review Title: " + df_merged['title_review'] + "\n" +
#             "Review Text: " + df_merged['text']
#         )
        
#         # 6. Clean up: Drop the redundant text columns to keep the file size small
#         df_final = df_merged.drop(columns=['title_review', 'text', 'title_meta', 'main_category', 'features'])
        
#         # 7. Write to the new Parquet file (using the .cast() trick to prevent schema errors!)
#         table = pa.Table.from_pandas(df_final)
        
#         if writer is None:
#             master_schema = table.schema
#             writer = pq.ParquetWriter(output_parquet, master_schema)
#         else:
#             table = table.cast(master_schema)
            
#         writer.write_table(table)
#         print(f"Merged and saved a chunk of {len(chunk)} reviews...")

#     if writer:
#         writer.close()
        
#     print(f"Success! Your hybrid data is safely saved at {output_parquet}")

# # --- How to run it ---
# # create_hybrid_parquet('cleaned_reviews.parquet', 'cleaned_meta.parquet', 'hybrid_documents.parquet')

In [31]:
# create_hybrid_parquet(cleaned_reviews_path, cleaned_meta_path, hybrid_parquet_path)

## 2. Convert parquet file to LangChain Document

In [32]:
# # simple tokenization function
# def simple_tokenize(text):
#     # Lowercase everything
#     text = text.lower()
#     # Remove everything except letters, numbers, spaces, and hyphens
#     text = re.sub(r"[^a-z0-9\s-]", "", text)
#     # Split into a list of words
#     return text.split()

In [33]:
# # 2. The Custom Lazy Parquet Loader
# def lazy_load_parquet(file_path, batch_size=10000):
#     """
#     Streams a Parquet file in chunks and yields LangChain Documents one by one.
#     This prevents the entire dataset from loading into RAM at once.
#     """
#     print(f"Streaming documents from {file_path}...")
#     parquet_file = pq.ParquetFile(file_path)
    
#     for batch in parquet_file.iter_batches(batch_size=batch_size):
#         # Convert just this small batch to pandas
#         df_chunk = batch.to_pandas()
        
#         # Yield one Document at a time
#         for _, row in df_chunk.iterrows():
#             content = row['page_content']
#             # Everything else in the row becomes the metadata dictionary
#             metadata = row.drop('page_content').to_dict()
            
#             yield Document(page_content=content, metadata=metadata)



In [34]:
# # 3. The Memory-Safe BM25 Builder
    
# # Initialize the generator (this uses almost zero RAM to start)
# document_generator = lazy_load_parquet(hybrid_parquet_path)

# # Pass the generator directly into the retriever!
# # It will pull documents one by one, tokenize them, and build the math model
# retriever = BM25Retriever.from_documents(
#     document_generator, 
#     preprocess_func=simple_tokenize
# )

# retriever.k = 3

# # Save the massive mathematical index to your hard drive
# with open(bm25_path, 'wb') as f:
#     pickle.dump(retriever, f)
    
# print(f"Index built and safely saved to {bm25_path}!")

In [35]:
# df_hybrid = pd.read_parquet(hybrid_parquet_path)

# # Set the page_content to be the column "title"
# loader = DataFrameLoader(df_hybrid, page_content_column="page_content")

# documents = loader.load()

# # Peek at the first document to see how it's structured
# print(documents[0].page_content)
# print(documents[0].metadata)

In [36]:
# retriever = BM25Retriever.from_documents(
#     documents, 
#     preprocess_func=simple_tokenize # swap out default tokenizer for our own.
# )

# # Bring back 3 results per search
# retriever.k = 3 

In [37]:
# # Run sample search query!
# query = "Green yarn"
# search_results = retriever.invoke(query)

# # View the 3 results
# print(f"\nTop {retriever.k} results for: '{query}'\n" + "-"*40)

# for i, doc in enumerate(search_results):
#     print(f"Rank {i+1}:")
#     # This is the main text (or title) we set as the page_content
#     print(f"Content: {doc.page_content}") 
#     # Pulling specific details out of the metadata dictionary
#     print(f"Description: {doc.metadata.get('description', 'Unknown')}")
#     print(f"Rating: {doc.metadata.get('average_rating', 'N/A')} Stars")
#     print("-" * 40)

### Save BM25 Index Using Pickle

In [38]:
# with open(index_file_path, 'wb') as f:
#     pickle.dump(retriever, f)
    
# print(f"Index built and saved to {index_file_path}!")



In [39]:
# def load_review_sample(filepath, n_rows=1000):
#     """
#     Loads a specific number of rows from a JSONL file and cleans the data.
#     """
#     print(f"Loading the first {n_rows} rows from {filepath}...")

#     # The nrows argument stops pandas from reading past our limit
#     df = pd.read_json(filepath, lines=True, nrows=n_rows)

#     print("Done! Data is ready for EDA.")
#     return df
